# 26. 토크나이저 직접 학습 — BPE 구현

> **제26장** · **이론편 대응: 20.2절 (토큰화)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (23장의 transformers 사용)
> **다운로드**: 없음

---

## 이 장에서 하는 일

23장에서 GPT-2 토크나이저를 **가져다 썼다.** 이번에는 **직접 만든다.**

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 왜 하위단어인가 | 20.2절 |
| 2 | **BPE 병합 과정 손계산 검증** ★ | 20.2절 |
| 3 | BPE 학습 구현 | 20.2절 |
| 4 | 인코딩·디코딩 구현 | 20.2절 |
| 5 | **한국어 토크나이저 학습** ★ | 20.2절 |
| 6 | 어휘 크기의 맞바꿈 | 20.2절 |
| 7 | 실제 라이브러리로 학습 | — |
| 8 | 특수 토큰과 후처리 | 24장 |

**2절과 5절이 핵심이다.** 이론편에서 손으로 계산한 병합 과정을 그대로 재현하고,
23장 4절에서 확인한 **한국어 토큰 비효율 문제**를 직접 해결해 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from collections import Counter, defaultdict
import re
import json
import time

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

print("준비 완료")

---

## 1. 왜 하위단어인가 — 이론편 20.2절

토큰화 방식은 세 가지로 나뉜다.

| 방식 | 예: "unhappiness" | 어휘 크기 | 문제 |
|---|---|---|---|
| 단어 단위 | `["unhappiness"]` | 수십만 | **모르는 단어를 못 다룸** |
| 문자 단위 | `["u","n","h",...]` | 수십~수백 | **너무 길어짐** |
| **하위단어** | `["un","happi","ness"]` | 수만 | 균형 |

**하위단어가 두 극단의 절충안**이다. 자주 쓰이는 것은 통째로, 드문 것은 조각으로 나눈다.

In [ ]:
import re
from collections import Counter

print("=" * 78)
print("세 가지 방식 비교")
print("=" * 78)

sample_texts = [
    "The unhappiness was unbelievable",
    "Tokenization is important",
    "Antidisestablishmentarianism",
]

# 작은 말뭉치로 단어 어휘를 만든다
mini_corpus = """
the cat sat on the mat
the dog ran in the park
a bird flew over the tree
the sun was bright today
""".strip().split()
word_vocab = set(mini_corpus)

print(f"학습 말뭉치의 단어 어휘: {len(word_vocab)}개")
print(f"  {sorted(word_vocab)}")
print()

print(f"{'방식':<14}{'입력':<34}{'결과'}")
print("-" * 78)

for text in sample_texts:
    words = text.lower().split()

    # 단어 단위
    word_result = [w if w in word_vocab else "<UNK>" for w in words]
    # 문자 단위
    char_result = list(text.replace(" ", "_"))

    print(f"\n{'원문':<14}{text}")
    print(f"{'  단어 단위':<14}{str(word_result)}")
    print(f"{'  문자 단위':<14}{len(char_result)}개 토큰: {char_result[:12]}...")

print()
print("-" * 78)
print("[단어 단위의 문제]")
print("  학습에 없던 단어가 전부 <UNK> 가 된다.")
print("  <UNK> 는 아무 정보가 없다 — 모델이 무엇인지 알 수 없다.")
print()
print("[문자 단위의 문제]")
print("  'Antidisestablishmentarianism' 이 28개 토큰이 된다.")
print("  문맥 창(이론편 20.4절)을 빠르게 소진하고, 학습도 어려워진다.")

---

## 2. BPE 병합 과정 ★ — 이론편 20.2절 검증

**Byte Pair Encoding**의 발상은 단순하다.

> **가장 자주 붙어 나오는 두 조각을 하나로 합친다. 이것을 반복한다.**

이론편 20.2절에서 손으로 계산한 말뭉치를 그대로 쓴다.

| 단어 | 빈도 |
|---|---|
| low | 5 |
| lower | 2 |
| newest | 6 |
| widest | 3 |

**이론편의 결과**: `(e, s)`가 9번으로 가장 빈번해 먼저 합쳐지고, 다음 라운드에 `est`가 된다.

In [ ]:
from collections import Counter

# 이론편 20.2절과 완전히 같은 말뭉치
BOOK_CORPUS = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

END_TOKEN = "</w>"      # 단어 끝 표시


def init_vocab(corpus):
    """단어를 문자로 쪼개고 끝 표시를 붙인다"""
    return {" ".join(list(word)) + " " + END_TOKEN: freq
            for word, freq in corpus.items()}


def get_pair_counts(vocab):
    """인접한 두 조각의 빈도를 센다"""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs


def merge_pair(pair, vocab):
    """한 쌍을 하나로 합친다"""
    merged = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in vocab.items():
        merged[word.replace(bigram, replacement)] = freq
    return merged


vocab = init_vocab(BOOK_CORPUS)

print("=" * 78)
print("초기 상태 — 문자 단위로 쪼갬")
print("=" * 78)
for word, freq in vocab.items():
    print(f"  {word:<28} x{freq}")
print()

# 첫 라운드의 쌍 빈도
pairs = get_pair_counts(vocab)
print("=" * 78)
print("인접 쌍의 빈도")
print("=" * 78)
print(f"{'쌍':<16}{'빈도':<10}{'계산 근거'}")
print("-" * 78)
for (a, b), count in pairs.most_common(8):
    sources = [f"{w}x{f}" for w, f in BOOK_CORPUS.items()
               if a + b in w or (a == b)]
    print(f"({a}, {b})".ljust(16) + f"{count:<10}")
print("-" * 78)
print()

best_pair, best_count = pairs.most_common(1)[0]
print(f"가장 빈번한 쌍: ({best_pair[0]}, {best_pair[1]}) — {best_count}번")
print()
print("이론편 20.2절: (e, s) 가 9번으로 최다")

assert best_pair == ("e", "s") and best_count == 9
print("[OK] 이론편과 일치")

In [ ]:
from collections import Counter

print("=" * 78)
print("병합 과정 — 6단계까지")
print("=" * 78)

vocab = init_vocab(BOOK_CORPUS)
merge_history = []

for step in range(1, 7):
    pairs = get_pair_counts(vocab)
    if not pairs:
        break

    top3 = pairs.most_common(3)
    best, count = top3[0]

    print(f"\n[{step}단계]")
    print("  상위 후보: " + ",  ".join(
        f"({a}{b})={c}" for (a, b), c in top3))
    print(f"  → 병합: ({best[0]}, {best[1]})  빈도 {count}")

    vocab = merge_pair(best, vocab)
    merge_history.append((best, count))

    for word, freq in vocab.items():
        print(f"      {word:<30} x{freq}")

print()
print("=" * 78)
print("병합 규칙 목록 (순서가 중요하다)")
print("=" * 78)
for i, (pair, count) in enumerate(merge_history, 1):
    print(f"  {i}. ({pair[0]}, {pair[1]})  →  {pair[0]+pair[1]}    (빈도 {count})")

print()
print("-" * 78)
# 이론편 값 검증
assert merge_history[0][0] == ("e", "s")
assert merge_history[1][0] == ("es", "t")
print("[OK] 이론편 20.2절과 일치")
print("     1단계: (e, s) → es")
print("     2단계: (es, t) → est   ← 이론편이 예고한 대로")
print()
print("[중요] 병합 순서가 규칙이 된다")
print("  나중에 새 텍스트를 인코딩할 때 **이 순서대로** 적용해야 한다.")
print("  순서가 바뀌면 다른 결과가 나온다 (4절에서 확인).")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 단계별 병합 빈도 ---
ax = axes[0]
steps = range(1, len(merge_history) + 1)
counts = [c for _, c in merge_history]
labels = [f"{p[0]}+{p[1]}" for p, _ in merge_history]

bars = ax.bar(steps, counts, color="#1E40AF")
for b, lab, c in zip(bars, labels, counts):
    ax.text(b.get_x() + b.get_width()/2, c + 0.15, lab,
            ha="center", fontsize=8)
ax.set_xlabel("병합 단계")
ax.set_ylabel("빈도")
ax.set_title("단계별 병합 대상과 빈도")
ax.set_xticks(list(steps))
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 토큰 수 변화 ---
ax = axes[1]
v = init_vocab(BOOK_CORPUS)
token_counts = [sum(len(w.split()) * f for w, f in v.items())]

for pair, _ in merge_history:
    v = merge_pair(pair, v)
    token_counts.append(sum(len(w.split()) * f for w, f in v.items()))

ax.plot(range(len(token_counts)), token_counts, marker="o",
        linewidth=2.5, color="#0D9488")
ax.set_xlabel("병합 횟수")
ax.set_ylabel("전체 토큰 수 (빈도 가중)")
ax.set_title("병합할수록 토큰이 줄어든다")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"병합 전: {token_counts[0]}개 토큰")
print(f"6번 병합 후: {token_counts[-1]}개 토큰 ({(1-token_counts[-1]/token_counts[0])*100:.0f}% 감소)")
print()
print("이것이 BPE 가 하는 일이다 — 자주 쓰이는 조각을 묶어 길이를 줄인다.")

---

## 3. BPE 학습 구현 — 이론편 20.2절

작은 예제로 원리를 확인했으니, **실제로 쓸 수 있는 형태**로 만든다.

In [ ]:
import re
from collections import Counter, defaultdict


class SimpleBPE:
    # BPE 토크나이저 (이론편 20.2절)
    #
    # 실제 구현과 다른 점:
    #   - 속도 최적화 없음 (원리 이해가 목적)
    #   - 바이트 수준 처리 대신 문자 수준
    # 같은 점:
    #   - 병합 규칙을 순서대로 학습하고 적용
    #   - 어휘 크기로 학습량 제어

    END = "</w>"

    def __init__(self):
        self.merges = []          # 병합 규칙 (순서 중요)
        self.vocab = {}           # 토큰 → ID
        self.inv_vocab = {}       # ID → 토큰

    def _word_freqs(self, texts):
        """말뭉치에서 단어 빈도를 센다"""
        counter = Counter()
        for text in texts:
            for word in re.findall(r"\w+|[^\w\s]", text.lower()):
                counter[word] += 1
        return counter

    def train(self, texts, vocab_size=300, verbose=False):
        """BPE 학습"""
        word_freqs = self._word_freqs(texts)

        # 문자 단위로 시작
        splits = {w: list(w) + [self.END] for w in word_freqs}

        # 기본 어휘: 등장하는 모든 문자
        alphabet = set()
        for w in word_freqs:
            alphabet.update(w)
        alphabet.add(self.END)

        vocab_tokens = sorted(alphabet)
        n_merges = max(0, vocab_size - len(vocab_tokens))

        for step in range(n_merges):
            # 쌍 빈도 계산
            pairs = Counter()
            for word, freq in word_freqs.items():
                syms = splits[word]
                for i in range(len(syms) - 1):
                    pairs[(syms[i], syms[i+1])] += freq

            if not pairs:
                break

            best, count = pairs.most_common(1)[0]
            if count < 2:          # 한 번만 나오면 병합할 가치가 없다
                break

            # 병합 적용
            new_token = best[0] + best[1]
            for word in splits:
                syms = splits[word]
                i = 0
                merged = []
                while i < len(syms):
                    if i < len(syms) - 1 and (syms[i], syms[i+1]) == best:
                        merged.append(new_token)
                        i += 2
                    else:
                        merged.append(syms[i])
                        i += 1
                splits[word] = merged

            self.merges.append(best)
            vocab_tokens.append(new_token)

            if verbose and (step + 1) % 50 == 0:
                print(f"    {step+1}번째 병합: ({best[0]}, {best[1]}) 빈도 {count}")

        self.vocab = {t: i for i, t in enumerate(vocab_tokens)}
        self.inv_vocab = {i: t for t, i in self.vocab.items()}
        return self

    def _tokenize_word(self, word):
        """한 단어에 병합 규칙을 순서대로 적용한다"""
        syms = list(word) + [self.END]
        for pair in self.merges:
            i = 0
            merged = []
            while i < len(syms):
                if i < len(syms) - 1 and (syms[i], syms[i+1]) == pair:
                    merged.append(pair[0] + pair[1])
                    i += 2
                else:
                    merged.append(syms[i])
                    i += 1
            syms = merged
        return syms

    def tokenize(self, text):
        """텍스트를 토큰 목록으로"""
        tokens = []
        for word in re.findall(r"\w+|[^\w\s]", text.lower()):
            tokens.extend(self._tokenize_word(word))
        return tokens

    def encode(self, text):
        """텍스트를 ID 목록으로"""
        return [self.vocab.get(t, -1) for t in self.tokenize(text)]

    def decode(self, ids):
        """ID 목록을 텍스트로"""
        tokens = [self.inv_vocab.get(i, "<UNK>") for i in ids]
        return "".join(tokens).replace(self.END, " ").strip()


print("SimpleBPE 정의 완료")
print()
print("[핵심 부분]")
print("  train()          : 빈도가 높은 쌍을 반복해 병합")
print("  _tokenize_word() : 학습한 규칙을 **순서대로** 적용")
print("  vocab_size       : 병합 횟수를 제어 (클수록 토큰이 짧아짐)")

In [ ]:
import time

# 영어 학습 말뭉치
english_corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is a method of data analysis",
    "Deep learning uses neural networks with many layers",
    "Natural language processing helps computers understand text",
    "The transformer architecture revolutionized language models",
    "Attention mechanisms allow models to focus on relevant parts",
    "Training large models requires significant computational resources",
    "Tokenization splits text into smaller meaningful units",
    "The model learns patterns from training data",
    "Neural networks consist of interconnected layers of neurons",
] * 20      # 반복해 빈도를 높인다

print("=" * 78)
print("영어 BPE 학습")
print("=" * 78)
print(f"말뭉치: {len(english_corpus)}문장")
print()

t0 = time.time()
bpe_en = SimpleBPE().train(english_corpus, vocab_size=200, verbose=True)
print(f"\n학습 완료: {time.time()-t0:.2f}초")
print(f"  어휘 크기 : {len(bpe_en.vocab)}")
print(f"  병합 규칙 : {len(bpe_en.merges)}개")
print()

print("학습된 병합 규칙 (앞 20개)")
print(f"{'순서':<8}{'병합':<20}{'결과 토큰'}")
print("-" * 78)
for i, (a, b) in enumerate(bpe_en.merges[:20], 1):
    print(f"{i:<8}({a}, {b})".ljust(28) + f"{a+b}")
print("-" * 78)
print()
print("자주 나오는 조각부터 합쳐진 것이 보인다.")

---

## 4. 인코딩·디코딩 — 이론편 20.2절

학습한 규칙으로 **새로운 텍스트**를 처리한다.

**핵심은 순서다.** 병합 규칙을 학습한 순서대로 적용해야 같은 결과가 나온다.

In [ ]:
print("=" * 78)
print("인코딩 결과")
print("=" * 78)

test_sentences = [
    "the model learns",                  # 학습에 있던 표현
    "deep learning is powerful",         # 일부만 있던 표현
    "unprecedented breakthrough",        # 학습에 없던 단어
]

for text in test_sentences:
    tokens = bpe_en.tokenize(text)
    ids = bpe_en.encode(text)
    decoded = bpe_en.decode(ids)

    print(f"\n원문   : {text}")
    print(f"토큰   : {tokens}")
    print(f"개수   : {len(tokens)}개 (글자 {len(text)}자)")
    print(f"복원   : {decoded}")
    print(f"일치   : {decoded.replace(' ', '') == text.replace(' ', '')}")

print()
print("=" * 78)
print("학습에 없던 단어도 처리된다")
print("=" * 78)
print("  'unprecedented' 는 말뭉치에 없지만 조각으로 나뉜다.")
print("  단어 단위 토크나이저라면 <UNK> 가 되었을 것이다.")
print()
print("  → 이론편 20.2절의 'OOV 문제 해결'이 이것이다.")

In [ ]:
print("=" * 78)
print("병합 순서가 왜 중요한가")
print("=" * 78)
print()

word = "learning"
print(f"단어: {word}")
print()

# 순서대로 적용하는 과정을 보여준다
syms = list(word) + [SimpleBPE.END]
print(f"  시작: {syms}")

applied = 0
for i, pair in enumerate(bpe_en.merges):
    before = list(syms)
    j = 0
    merged = []
    changed = False
    while j < len(syms):
        if j < len(syms) - 1 and (syms[j], syms[j+1]) == pair:
            merged.append(pair[0] + pair[1])
            j += 2
            changed = True
        else:
            merged.append(syms[j])
            j += 1
    syms = merged
    if changed:
        applied += 1
        print(f"  규칙 {i+1:>3} ({pair[0]}, {pair[1]}) 적용 → {syms}")
    if applied >= 8:
        break

print(f"  ...")
print(f"  최종: {bpe_en.tokenize(word)}")
print()
print("-" * 78)
print("[순서를 바꾸면]")
print("  예를 들어 (n, g) 를 먼저 합치면 'ng' 가 생겨")
print("  나중에 (i, ng) 규칙이 적용될 기회가 사라질 수 있다.")
print()
print("  → 학습 시의 순서를 그대로 저장하고 적용해야 한다")
print("  → 토크나이저 파일에 merges.txt 가 들어 있는 이유다")

---

## 5. 한국어 토크나이저 학습 ★ — 이론편 20.2절

**23장 4절에서 확인한 문제를 기억하는가.**

GPT-2 토크나이저는 한국어를 글자당 2.12토큰으로 쪼갰다. 영어의 13배였다.
어휘에 한글 조각이 거의 없었기 때문이다.

**한국어로 학습시키면 어떻게 달라질까.**

In [ ]:
import time

korean_corpus = [
    "인공지능은 인간의 지능을 모방하는 기술이다",
    "머신러닝은 데이터로부터 규칙을 학습하는 방법이다",
    "딥러닝은 여러 층의 신경망을 사용하는 머신러닝의 한 갈래다",
    "자연어처리는 컴퓨터가 인간의 언어를 이해하도록 돕는다",
    "트랜스포머 구조는 언어모델의 발전을 이끌었다",
    "어텐션 기법은 중요한 부분에 집중하도록 한다",
    "대규모 모델을 학습하려면 많은 계산 자원이 필요하다",
    "토큰화는 문장을 의미 있는 단위로 나누는 과정이다",
    "모델은 학습 데이터에서 패턴을 배운다",
    "신경망은 서로 연결된 층으로 구성된다",
    "학습률은 모델이 얼마나 빠르게 배울지를 결정한다",
    "과대적합은 학습 데이터만 잘 맞히는 현상이다",
    "정규화는 과대적합을 줄이는 여러 방법을 말한다",
    "검색 증강 생성은 외부 문서를 참고해 답한다",
    "파인튜닝은 사전학습된 모델을 특정 작업에 맞춘다",
] * 30

print("=" * 78)
print("한국어 BPE 학습")
print("=" * 78)
print(f"말뭉치: {len(korean_corpus)}문장")
print()

t0 = time.time()
bpe_ko = SimpleBPE().train(korean_corpus, vocab_size=500)
print(f"학습 완료: {time.time()-t0:.2f}초")
print(f"  어휘 크기: {len(bpe_ko.vocab)}")
print(f"  병합 규칙: {len(bpe_ko.merges)}개")
print()

print("학습된 병합 규칙 (앞 25개)")
print(f"{'순서':<8}{'병합 결과'}")
print("-" * 78)
for i, (a, b) in enumerate(bpe_ko.merges[:25], 1):
    print(f"{i:<8}{a} + {b}  →  {a+b}")
print("-" * 78)
print()
print("'학습', '모델', '데이터' 같은 자주 쓰이는 조각이 하나로 묶인다.")

In [ ]:
from transformers import AutoTokenizer

print("=" * 78)
print("GPT-2 vs 직접 학습한 한국어 BPE")
print("=" * 78)
print()

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

test_korean = [
    "인공지능은 데이터를 학습한다",
    "딥러닝 모델의 파라미터",
    "자연어처리 기술의 발전",
]

print(f"{'문장':<30}{'글자':<8}{'GPT-2':<12}{'직접 학습':<12}{'개선'}")
print("-" * 78)

gpt2_counts, ours_counts, char_counts = [], [], []

for text in test_korean:
    n_char = len(text)
    n_gpt2 = len(gpt2_tok.encode(text))
    n_ours = len(bpe_ko.tokenize(text))

    char_counts.append(n_char)
    gpt2_counts.append(n_gpt2)
    ours_counts.append(n_ours)

    improve = (1 - n_ours / n_gpt2) * 100
    print(f"{text[:28]:<30}{n_char:<8}{n_gpt2:<12}{n_ours:<12}{improve:>5.0f}%")

print("-" * 78)
print()

total_char = sum(char_counts)
total_gpt2 = sum(gpt2_counts)
total_ours = sum(ours_counts)

print(f"{'':<20}{'총 토큰':<14}{'글자당 토큰'}")
print("-" * 78)
print(f"{'GPT-2':<20}{total_gpt2:<14}{total_gpt2/total_char:.2f}")
print(f"{'직접 학습':<20}{total_ours:<14}{total_ours/total_char:.2f}")
print("-" * 78)
print()
print(f"토큰 수가 {(1-total_ours/total_gpt2)*100:.0f}% 줄었다.")
print()
print("[이것이 뜻하는 것]")
print("  같은 문장을 더 적은 토큰으로 표현한다")
print("  → 문맥 창을 덜 차지한다 (이론편 20.4절)")
print("  → API 비용이 줄어든다 (25장)")
print("  → 생성 속도가 빨라진다")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 토큰 수 비교 ---
ax = axes[0]
x = np.arange(len(test_korean))
w = 0.36
ax.bar(x - w/2, gpt2_counts, w, label="GPT-2", color="#DC2626")
ax.bar(x + w/2, ours_counts, w, label="직접 학습", color="#0D9488")
for i, (g, o) in enumerate(zip(gpt2_counts, ours_counts)):
    ax.text(i - w/2, g + 0.5, str(g), ha="center", fontsize=8)
    ax.text(i + w/2, o + 0.5, str(o), ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([f"문장 {i+1}" for i in range(len(test_korean))], fontsize=9)
ax.set_ylabel("토큰 수")
ax.set_title("한국어 문장의 토큰 수")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 토큰 분해 예시 ---
ax = axes[1]
sample = test_korean[0]
gpt2_tokens = [gpt2_tok.decode([i]) for i in gpt2_tok.encode(sample)]
our_tokens = [t.replace("</w>", "_") for t in bpe_ko.tokenize(sample)]

ax.barh([1], [len(gpt2_tokens)], color="#DC2626", height=0.4)
ax.barh([0], [len(our_tokens)], color="#0D9488", height=0.4)
ax.set_yticks([0, 1])
ax.set_yticklabels(["직접 학습", "GPT-2"])
ax.set_xlabel("토큰 수")
ax.set_title(f"'{sample[:16]}...' 분해")
ax.text(len(gpt2_tokens) + 0.3, 1, f"{len(gpt2_tokens)}개", va="center", fontsize=10)
ax.text(len(our_tokens) + 0.3, 0, f"{len(our_tokens)}개", va="center", fontsize=10)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("실제 분해 결과")
print(f"  GPT-2    : {[t if t.strip() else '_' for t in gpt2_tokens[:14]]}...")
print(f"  직접 학습 : {our_tokens[:14]}")
print()
print("GPT-2 는 알아볼 수 없는 조각들이다 (UTF-8 바이트 단위).")
print("직접 학습한 것은 의미 있는 한글 조각으로 나뉜다.")

---

## 6. 어휘 크기의 맞바꿈 — 이론편 20.2절

**어휘를 크게 하면 토큰이 짧아진다.** 그런데 왜 무한정 키우지 않을까.

| 어휘 크기 | 토큰 길이 | 임베딩 크기 | 출력층 |
|---|---|---|---|
| 작음 | 길어짐 | 작음 | 작음 |
| 큼 | 짧아짐 | **큼** | **큼** |

23장 5절에서 봤듯 **임베딩과 출력층은 어휘 크기에 비례**한다.

In [ ]:
import time
import numpy as np

print("=" * 78)
print("어휘 크기에 따른 변화")
print("=" * 78)
print("(각 크기마다 학습하므로 30초 정도 걸립니다)")
print()

vocab_sizes = [100, 200, 400, 800, 1500]
eval_texts = test_korean + [
    "모델 학습에 필요한 데이터",
    "신경망의 층과 파라미터",
]

results_vocab = []
for vs in vocab_sizes:
    t0 = time.time()
    bpe = SimpleBPE().train(korean_corpus, vocab_size=vs)
    elapsed = time.time() - t0

    total_tokens = sum(len(bpe.tokenize(t)) for t in eval_texts)
    total_chars = sum(len(t) for t in eval_texts)

    results_vocab.append({
        "vocab": len(bpe.vocab),
        "merges": len(bpe.merges),
        "tokens": total_tokens,
        "ratio": total_tokens / total_chars,
        "time": elapsed,
    })

print(f"{'목표 어휘':<12}{'실제 어휘':<12}{'병합 규칙':<12}"
      f"{'총 토큰':<12}{'글자당':<12}{'학습 시간'}")
print("-" * 78)
for vs, r in zip(vocab_sizes, results_vocab):
    print(f"{vs:<12}{r['vocab']:<12}{r['merges']:<12}"
          f"{r['tokens']:<12}{r['ratio']:<12.3f}{r['time']:.2f}초")
print("-" * 78)
print()

# 모델 크기에 미치는 영향
print("어휘 크기가 모델에 미치는 영향 (d_model=768 가정)")
print(f"{'어휘 크기':<14}{'임베딩 파라미터':<20}{'출력층 파라미터':<20}{'합계'}")
print("-" * 78)
d_model = 768
for vs in [1000, 10000, 32000, 50000, 100000]:
    emb = vs * d_model
    out = d_model * vs
    print(f"{vs:<14,}{emb:<20,}{out:<20,}{emb+out:,}")
print("-" * 78)
print()
print("어휘 5만이면 임베딩과 출력층만 7,680만 파라미터다.")
print("  23장에서 GPT-2 의 파라미터 분포를 봤을 때 임베딩이 컸던 이유다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 어휘 크기와 토큰 효율 ---
ax = axes[0]
vocabs = [r["vocab"] for r in results_vocab]
ratios = [r["ratio"] for r in results_vocab]

ax.plot(vocabs, ratios, marker="o", linewidth=2.5, color="#1E40AF")
for v, r in zip(vocabs, ratios):
    ax.text(v, r + 0.02, f"{r:.2f}", ha="center", fontsize=8)
ax.set_xlabel("어휘 크기")
ax.set_ylabel("글자당 토큰 수")
ax.set_title("어휘가 크면 토큰이 짧아진다")
ax.grid(alpha=0.3)

# --- 오른쪽: 어휘 크기와 파라미터 ---
ax = axes[1]
vs_range = np.array([1000, 5000, 10000, 32000, 50000, 100000, 200000])
d_model = 768
params = vs_range * d_model * 2 / 1e6

ax.plot(vs_range, params, marker="s", linewidth=2.5, color="#EA580C")
ax.axvline(50257, color="#0D9488", linestyle="--", linewidth=1.5)
ax.text(52000, params.max() * 0.5, "GPT-2\n50,257", fontsize=8, color="#0D9488")
ax.set_xscale("log")
ax.set_xlabel("어휘 크기 (로그)")
ax.set_ylabel("임베딩+출력층 (백만 파라미터)")
ax.set_title("어휘가 크면 모델도 커진다")
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("=" * 78)
print("실제 모델들의 어휘 크기")
print("=" * 78)
print(f"{'모델 계열':<24}{'어휘 크기':<16}{'특징'}")
print("-" * 78)
print(f"{'GPT-2':<24}{'50,257':<16}영어 중심")
print(f"{'BERT (base)':<24}{'30,522':<16}WordPiece")
print(f"{'다국어 모델':<24}{'100,000~250,000':<16}여러 언어 포함")
print(f"{'최근 LLM':<24}{'32,000~200,000':<16}모델마다 다름")
print("-" * 78)
print()
print("[균형점]")
print("  너무 작으면: 토큰이 길어져 문맥 창 소모, 학습 어려움")
print("  너무 크면  : 모델이 커지고, 드문 토큰은 잘 학습되지 않음")
print()
print("  보통 3만~10만 사이에서 정한다.")

---

## 7. 실제 라이브러리로 학습 — Hugging Face `tokenizers`

직접 구현해 원리를 확인했으니, **실무에서 쓰는 방법**을 본다.

`transformers`와 함께 설치되는 `tokenizers` 라이브러리를 쓴다.
Rust로 구현되어 **훨씬 빠르다.**

In [ ]:
print("=" * 78)
print("tokenizers 라이브러리 사용법")
print("=" * 78)
print()

try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    HAS_TOKENIZERS = True
    import tokenizers
    print(f"tokenizers {tokenizers.__version__} 확인")
except ImportError:
    HAS_TOKENIZERS = False
    print("[없음] tokenizers 미설치")
    print("  transformers 를 설치하면 함께 들어옵니다.")

print()
print("코드 형태")
print()
example = [
    "from tokenizers import Tokenizer, models, trainers, pre_tokenizers",
    "",
    "# 1) 빈 BPE 토크나이저",
    "tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))",
    "",
    "# 2) 사전 분할 방식 (공백 기준 등)",
    "tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)",
    "",
    "# 3) 학습 설정",
    "trainer = trainers.BpeTrainer(",
    "    vocab_size=30000,",
    "    special_tokens=['[UNK]', '[CLS]', '[SEP]', '[PAD]', '[MASK]'],",
    "    min_frequency=2,",
    ")",
    "",
    "# 4) 학습 (파일 또는 이터레이터)",
    "tokenizer.train_from_iterator(corpus, trainer)",
    "",
    "# 5) 저장",
    "tokenizer.save('my_tokenizer.json')",
]
for line in example:
    print("  " + line)

In [ ]:
import time

if HAS_TOKENIZERS:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

    print("=" * 78)
    print("실제로 학습해 보기")
    print("=" * 78)

    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

    trainer = trainers.BpeTrainer(
        vocab_size=500,
        special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
        min_frequency=2,
    )

    t0 = time.time()
    tokenizer.train_from_iterator(korean_corpus, trainer)
    lib_time = time.time() - t0

    print(f"  학습 시간: {lib_time:.3f}초")
    print(f"  어휘 크기: {tokenizer.get_vocab_size()}")
    print()

    # 우리 구현과 속도 비교
    t0 = time.time()
    _ = SimpleBPE().train(korean_corpus, vocab_size=500)
    ours_time = time.time() - t0

    print(f"{'구현':<20}{'학습 시간':<16}{'배수'}")
    print("-" * 78)
    print(f"{'직접 구현 (Python)':<20}{ours_time:<16.3f}{ours_time/lib_time:.0f}배 느림")
    print(f"{'tokenizers (Rust)':<20}{lib_time:<16.3f}기준")
    print("-" * 78)
    print()

    print("인코딩 결과 비교")
    for text in test_korean[:2]:
        enc = tokenizer.encode(text)
        ours = bpe_ko.tokenize(text)
        print(f"\n  원문      : {text}")
        print(f"  라이브러리 : {enc.tokens[:12]}")
        print(f"  직접 구현  : {[t.replace('</w>','_') for t in ours[:12]]}")

    print()
    print("-" * 78)
    print("결과가 조금 다르다.")
    print("  사전 분할 방식, 특수 토큰 처리, 최적화 세부가 다르기 때문이다.")
    print("  원리는 같다 — 빈도 높은 쌍을 반복 병합")
else:
    print("=" * 78)
    print("(tokenizers 미설치 — 코드 형태만 확인)")
    print("=" * 78)
    print()
    print("실무에서 직접 구현을 쓰지 않는 이유")
    print(f"  {'항목':<20}{'직접 구현':<20}{'라이브러리'}")
    print("-" * 78)
    print(f"  {'속도':<20}{'느림 (Python)':<20}{'빠름 (Rust)'}")
    print(f"  {'대용량 처리':<20}{'메모리 부담':<20}{'스트리밍 지원'}")
    print(f"  {'정규화':<20}{'직접 구현':<20}{'NFD/NFC 등 내장'}")
    print(f"  {'호환성':<20}{'별도 처리':<20}{'transformers 연동'}")
    print("-" * 78)

---

## 8. 특수 토큰과 후처리 — 24장과 이어짐

토크나이저는 텍스트를 나누는 것만 하지 않는다.
**24장에서 다룬 대화 형식**도 토크나이저가 처리한다.

In [ ]:
print("=" * 78)
print("특수 토큰의 역할")
print("=" * 78)
print()
print(f"{'토큰':<16}{'용도':<34}{'어디서 봤나'}")
print("-" * 78)
special = [
    ("[UNK]", "어휘에 없는 것", "1절 단어 단위 토큰화"),
    ("[PAD]", "길이 맞추기용 채움", "22장 4절 배치 처리"),
    ("[BOS] / <s>", "문장 시작", "22장 5절 생성"),
    ("[EOS] / </s>", "문장 끝", "31장 SFT — '답을 마치는 법'"),
    ("[CLS]", "문장 전체를 대표", "BERT 계열 분류"),
    ("[SEP]", "문장 구분", "BERT 계열 문장 쌍"),
    ("[MASK]", "가려진 토큰", "BERT 사전학습"),
    ("<|im_start|>", "대화 역할 시작", "24장 5절 ChatML"),
]
for a, b, c in special:
    print(f"{a:<16}{b:<34}{c}")
print("-" * 78)
print()
print("[특수 토큰이 어휘에 있어야 하는 이유]")
print("  모델은 토큰 ID 만 본다. 특수 토큰도 하나의 ID 여야 한다.")
print("  일반 텍스트와 겹치지 않도록 대괄호나 <| |> 같은 형태를 쓴다.")

In [ ]:
from transformers import AutoTokenizer

print("=" * 78)
print("실제 토크나이저의 특수 토큰")
print("=" * 78)

tok = AutoTokenizer.from_pretrained("gpt2")

print(f"{'속성':<24}{'값':<20}{'ID'}")
print("-" * 78)
for name in ["bos_token", "eos_token", "unk_token", "pad_token"]:
    val = getattr(tok, name, None)
    tid = getattr(tok, name + "_id", None)
    print(f"{name:<24}{str(val):<20}{tid}")
print("-" * 78)
print()
print("GPT-2 는 pad_token 이 없다.")
print("  생성 전용 모델이라 패딩이 필요 없었기 때문이다.")
print("  22장·28장에서 tokenizer.pad_token = tokenizer.eos_token 을 쓴 이유다.")
print()

# 저장 형식 확인
print("=" * 78)
print("토크나이저 파일 구성")
print("=" * 78)
print()
print(f"{'파일':<26}{'내용'}")
print("-" * 78)
files = [
    ("vocab.json", "토큰 → ID 대응표"),
    ("merges.txt", "병합 규칙 (순서대로!)"),
    ("tokenizer.json", "위 둘을 합친 형식 (최근)"),
    ("tokenizer_config.json", "설정 (특수 토큰, 대화 템플릿 등)"),
    ("special_tokens_map.json", "특수 토큰 정의"),
]
for a, b in files:
    print(f"{a:<26}{b}")
print("-" * 78)
print()
print("merges.txt 가 4절에서 강조한 **병합 순서**를 담고 있다.")
print("  이 파일이 없으면 같은 어휘라도 다르게 쪼개진다.")

In [ ]:
import json
from pathlib import Path

print("=" * 78)
print("직접 만든 토크나이저 저장하고 불러오기")
print("=" * 78)

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
save_dir = root / "outputs" / "my_tokenizer"
save_dir.mkdir(parents=True, exist_ok=True)


def save_bpe(bpe, directory):
    """어휘와 병합 규칙을 저장한다"""
    directory = Path(directory)
    with open(directory / "vocab.json", "w", encoding="utf-8") as f:
        json.dump(bpe.vocab, f, ensure_ascii=False, indent=1)
    with open(directory / "merges.txt", "w", encoding="utf-8") as f:
        f.write("# version: simple-bpe\n")
        for a, b in bpe.merges:
            f.write(f"{a} {b}\n")
    return directory


def load_bpe(directory):
    """저장한 것을 불러온다"""
    directory = Path(directory)
    bpe = SimpleBPE()
    with open(directory / "vocab.json", encoding="utf-8") as f:
        bpe.vocab = json.load(f)
    bpe.inv_vocab = {v: k for k, v in bpe.vocab.items()}
    bpe.merges = []
    with open(directory / "merges.txt", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split(" ")
            if len(parts) == 2:
                bpe.merges.append((parts[0], parts[1]))
    return bpe


save_bpe(bpe_ko, save_dir)
print(f"저장 위치: {save_dir}")
for f in sorted(save_dir.glob("*")):
    print(f"  {f.name:<20}{f.stat().st_size:>8,} 바이트")
print()

# 불러와서 같은 결과가 나오는지 확인
loaded = load_bpe(save_dir)
test = test_korean[0]

original_tokens = bpe_ko.tokenize(test)
loaded_tokens = loaded.tokenize(test)

print(f"원본 토크나이저: {original_tokens[:10]}")
print(f"불러온 토크나이저: {loaded_tokens[:10]}")
print(f"일치: {original_tokens == loaded_tokens}")
assert original_tokens == loaded_tokens
print()
print("[OK] 저장과 복원이 정확히 동작한다")
print()
print("실무에서는 tokenizer.save_pretrained() / from_pretrained() 를 쓴다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **20.2** | **(e, s) 빈도 9로 최다** | **일치** ✓ |
| **20.2** | **2단계에서 `est` 형성** | **일치** ✓ |
| 20.2 | 하위단어로 OOV 해결 | 확인 ✓ |
| 20.2 | 어휘 크기의 맞바꿈 | 측정 ✓ |

### BPE 알고리즘 요약

```
1. 문자 단위로 시작
2. 반복:
     - 인접 쌍의 빈도를 센다
     - 가장 빈번한 쌍을 하나로 합친다
     - 병합 규칙을 순서대로 기록
3. 어휘 크기에 도달하면 중단
```

**인코딩할 때는 학습한 순서대로** 병합 규칙을 적용한다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| 하위단어의 이점 | OOV 해결 + 적당한 길이 |
| **병합 순서** | **바뀌면 결과가 달라짐 → merges.txt** |
| 한국어 토크나이저 | 직접 학습하면 토큰 수 크게 감소 |
| 어휘 크기 | 크면 토큰↓ 모델↑ — 보통 3만~10만 |
| 임베딩 파라미터 | 어휘 크기 × d_model × 2 |
| `</w>` | 단어 끝 표시 — 복원에 필요 |
| 특수 토큰 | 어휘에 포함되어야 함 |
| 실무 | `tokenizers` 라이브러리 (Rust, 훨씬 빠름) |

### 23장과 이어지는 지점

23장 4절에서 **한국어가 영어보다 13배 많은 토큰**을 쓴다는 것을 봤다.
이 장에서 그 원인(어휘에 한글 조각이 없음)과 해법(직접 학습)을 확인했다.

**다만 실무에서는** 토크나이저만 바꿀 수 없다. 모델이 그 토크나이저로 학습되었기 때문이다.
한국어를 잘 다루려면 **한국어를 포함해 사전학습된 모델**을 골라야 한다.

### 다음 장

**27. Embedding과 벡터 검색** — 토큰이 벡터가 되는 다음 단계다.
이 장에서 나눈 조각들이 어떻게 의미를 갖게 되는지 다룬다.